In [5]:
"""Pydantic Perfomance testing."""


import json
import timeit
from urllib.parse import urlparse
import requests
from pydantic import HttpUrl, TypeAdapter

REPOS = 7
NUMBER= 100
r = requests.get('https://api.github.com/emojis', timeout=10)
r.raise_for_status()
emojis_json = r.content

def emojis_pure_python(raw_data):
    """Parse emojis JSON."""
    data = json.loads(raw_data)
    output = {}
    for key, value in data.items():
        assert isinstance(key, str)
        url = urlparse(value)
        assert url.scheme in ('http', 'https')
        output[key] = url

emojis_pure_python_times = timeit.repeat(
    'emojis_pure_python(emojis_json)',
    globals={
        'emojis_pure_python': emojis_pure_python,
        'emojis_json': emojis_json,
    },
    repeat=REPOS,
    number=NUMBER,
)
print(f'pure python: {min(emojis_pure_python_times) / NUMBER * 1000:0.2f}ms')

type_adapter = TypeAdapter(dict[str, HttpUrl])
emojis_pydantic_times = timeit.repeat(
    'type_adapter.validate_json(emojis_json)',
    globals={
        'type_adapter': type_adapter,
        'HttpUrl': HttpUrl,
        'emojis_json': emojis_json,
    },
    repeat=REPOS,
    number=NUMBER,
)

print(f"pydantic: {min(emojis_pydantic_times) / NUMBER * 1000:0.2f}ms")

print(f'Pydantic {min(emojis_pure_python_times) / min(emojis_pydantic_times):0.2f}x faster')

pure python: 2.60ms
pydantic: 1.03ms
Pydantic 2.53x faster


In [ ]:
# Pydantic Serialization 3 ways

from datetime import datetime
from pydantic import BaseModel

class Meeting(BaseModel):
    """Meeting model."""

    when: datetime
    where: bytes
    why: str = 'No idea'

m = Meeting(when="2020-01-01T12:00", where="home")
print(m.model_dump(exclude_unset=True))
print(m.model_dump(exclude={'where'}, mode='json'))
print(m.model_dump_json(exclude_defaults=True))


{'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}
{'when': '2020-01-01T12:00:00', 'why': 'No idea'}
{"when":"2020-01-01T12:00:00","where":"home"}


In [1]:
# Json Schema. can be generated for any Pydantic schema

from datetime import datetime
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    zipcode: str

class Meeting(BaseModel):
    when: datetime
    where: Address
    why: str = 'No idea'

print(Meeting.model_json_schema())

{'$defs': {'Address': {'properties': {'street': {'title': 'Street', 'type': 'string'}, 'city': {'title': 'City', 'type': 'string'}, 'zipcode': {'title': 'Zipcode', 'type': 'string'}}, 'required': ['street', 'city', 'zipcode'], 'title': 'Address', 'type': 'object'}}, 'properties': {'when': {'format': 'date-time', 'title': 'When', 'type': 'string'}, 'where': {'$ref': '#/$defs/Address'}, 'why': {'default': 'No idea', 'title': 'Why', 'type': 'string'}}, 'required': ['when', 'where'], 'title': 'Meeting', 'type': 'object'}


In [1]:
# Dataclasses, TypedDict, and more.
from datetime import datetime
from typing_extensions import NotRequired, TypedDict
from pydantic import TypeAdapter


class Meeting(TypedDict):
    """Meeting model."""
    when: datetime
    where: bytes
    why: NotRequired[str]

meeting_adapter = TypeAdapter(Meeting)
m = meeting_adapter.validate_python(
    {
        "when": "2020-01-01T12:00",
        "where": b"123 Main St",
    }
)
print(m)
print(m["when"])
meeting_adapter.dump_python(m, exclude={'where'})
print(meeting_adapter.json_schema())

{'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'123 Main St'}
2020-01-01 12:00:00
{'description': 'Meeting model.', 'properties': {'when': {'format': 'date-time', 'title': 'When', 'type': 'string'}, 'where': {'format': 'binary', 'title': 'Where', 'type': 'string'}, 'why': {'title': 'Why', 'type': 'string'}}, 'required': ['when', 'where'], 'title': 'Meeting', 'type': 'object'}
